# MovieLens Latest Small Dataset (`ml-latest-small`)

## Overview
The **MovieLens Latest Small** dataset is a benchmark dataset collected and maintained by **GroupLens Research** at the University of Minnesota. It describes user rating and free-text tagging activity from MovieLens, a non-commercial web-based movie recommendation service.

This dataset is ideal for prototyping **Content-Based**, **Collaborative Filtering**, and **Hybrid Recommendation Engines** due to its high interaction density and lack of structural noise.


---

## Dataset Quick Statistics
* **Total Ratings:** ~100,836 ratings
* **Total Tag Applications:** ~3,683 tags
* **Total Movies:** 9,742 movies
* **Total Users:** 610 users
* **Rating Scale:** 0.5 to 5.0 stars (in 0.5 increments)
* **Time Range:** March 29, 1996 – September 24, 2018

---

## File Structure & Feature Breakdown

The dataset comes as a `.zip` file containing **4 separate CSV files**. All files are UTF-8 encoded and formatted with standard comma delimiters.

### 1. `movies.csv` (Content Metadata)
Contains metadata for all movies available in the dataset. Used primarily for **Content-Based Filtering (TF-IDF)**.

| Column | Data Type | Description | Example |
| :--- | :--- | :--- | :--- |
| **`movieId`** | Integer | Unique identifier for each movie. | `1` |
| **`title`** | String | Full movie title, including release year in parentheses. | `Toy Story (1995)` |
| **`genres`** | String | Pipe-separated list of genres (`\|`). | `Adventure\|Animation\|Children\|Comedy\|Fantasy` |

* **Special Value:** If a movie has no categorized genre, it is marked as `(no genres listed)`.
* **Available Genres:** Action, Adventure, Animation, Children, Comedy, Crime, Documentary, Drama, Fantasy, Film-Noir, Horror, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western.

---

### 2. `ratings.csv` (User Interactions)
Contains explicit rating logs provided by users. Used primarily for **Collaborative Filtering (SVD / Matrix Factorization)**.

| Column | Data Type | Description | Example |
| :--- | :--- | :--- | :--- |
| **`userId`** | Integer | Unique identifier for each user (anonymized). | `1` |
| **`movieId`** | Integer | Unique identifier matching `movies.csv`. | `1` |
| **`rating`** | Float | Star rating given by the user ($0.5$ to $5.0$). | `4.0` |
| **`timestamp`** | Integer | UTC timestamp (seconds since midnight Jan 1, 1970). | `964982703` |

* **User Privacy & Sparsity:** Every user in this dataset has rated **at least 20 movies**.

---

### 3. `tags.csv` (User-Generated Keywords)
Contains free-text metadata applied to movies by users. Excellent for enriching **TF-IDF features**.

| Column | Data Type | Description | Example |
| :--- | :--- | :--- | :--- |
| **`userId`** | Integer | ID of the user who created the tag. | `2` |
| **`movieId`** | Integer | ID of the movie being tagged. | `60756` |
| **`tag`** | String | Short user-entered keyword or phrase. | `funny` |
| **`timestamp`** | Integer | UTC timestamp when the tag was submitted. | `1445712298` |

---

### 4. `links.csv` (External Identifiers)
Maps MovieLens IDs to external movie databases for fetching external posters or rich plot details.

| Column | Data Type | Description | Example |
| :--- | :--- | :--- | :--- |
| **`movieId`** | Integer | MovieLens unique ID. | `1` |
| **`imdbId`** | String/Int | IMDb identifier (can construct `https://www.imdb.com/title/tt{imdbId}/`). | `0114709` |
| **`tmdbId`** | Integer | The Movie Database (TMDb) identifier. | `862` |

---

## How to Apply ML Techniques to MovieLens

```
                         ┌───► Content-Based (TF-IDF) ───► Feature: `genres` + `tag`
Input Query / User ID ───┤
                         └───► Collaborative (SVD)   ───► Pivot: `userId` x `movieId`
```

### 1. Content-Based Filtering (TF-IDF + Cosine Similarity)
* **Features:** Concatenate `genres` and user-generated `tag` words into a single text vector.
* **Method:** Compute Cosine Similarity between movie vectors.
* **Use Case:** Recommending items similar to a specific movie without requiring user history.

### 2. Collaborative Filtering (SVD / TruncatedSVD)
* **Features:** Pivot `ratings.csv` into a sparse interaction matrix $R$ of shape $(\text{Users} \times \text{Movies})$.
* **Method:** Apply Singular Value Decomposition ($R \approx U \cdot \Sigma \cdot V^T$) to predict unobserved ratings.
* **Use Case:** Personalizing recommendations for existing users based on past ratings.

---

## Python Loading Snippet

```python
import pandas as pd

# 1. Load files
movies = pd.read_csv("ml-latest-small/movies.csv")
ratings = pd.read_csv("ml-latest-small/ratings.csv")
tags = pd.read_csv("ml-latest-small/tags.csv")

# 2. Extract Release Year from Title
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')
movies['title_clean'] = movies['title

In [1]:
import pandas as pd 
import numpy as np 

In [2]:
movies = pd.read_csv(r'../datasets/movies.csv')
ratings = pd.read_csv(r'../datasets/ratings.csv')
tags = pd.read_csv(r'../datasets/tags.csv')

## Movies

In [41]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [42]:
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').copy()

In [43]:
movies.head()

,movieId,title,genres,year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men (1995),Comedy|Romance,1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II (1995),Comedy,1995


In [44]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
 3   year     9729 non-null   str  
dtypes: int64(1), str(3)
memory usage: 304.6 KB


## Ratings

In [45]:
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [46]:
ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [47]:
# first we will Convert Unix timestamp to datetime
ratings['timestamp_dt'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['rating_year'] = ratings['timestamp_dt'].dt.year
ratings

,userId,movieId,rating,timestamp,timestamp_dt,rating_year
0,1,1,4.0,964982703,2000-07-30 18:45:03,2000
1,1,3,4.0,964981247,2000-07-30 18:20:47,2000
2,1,6,4.0,964982224,2000-07-30 18:37:04,2000
3,1,47,5.0,964983815,2000-07-30 19:03:35,2000
4,1,50,5.0,964982931,2000-07-30 18:48:51,2000
...,...,...,...,...,...,...
100831,610,166534,4.0,1493848402,2017-05-03 21:53:22,2017
100832,610,168248,5.0,1493850091,2017-05-03 22:21:31,2017
100833,610,168250,5.0,1494273047,2017-05-08 19:50:47,2017
100834,610,168252,5.0,1493846352,2017-05-03 21:19:12,2017


# Popular Movies Recommender data set

In [48]:
# Getting number of ratings applied to each movie and avg_rating applied to each movie by users
popular_movies = ratings.groupby('movieId').agg(
    num_ratings = ('rating','count'),
    avg_rating = ('rating','mean')
).reset_index().copy()
    

In [49]:
# keeping only those movieId's where num_ratings is greater than 200 and avg rating is greater than 3
popular_movies = popular_movies.query('num_ratings >= 150 and avg_rating >= 2.8').copy()

In [52]:
popular_movies = popular_movies.sort_values('avg_rating',ascending=False).copy()

In [54]:
popular_movies_final = pd.merge(popular_movies,movies,on='movieId')

In [56]:
popular_movies_final

,movieId,num_ratings,avg_rating,title,genres,year
0,318,317,4.429022,"Shawshank Redemption, The (1994)",Crime|Drama,1994
1,858,192,4.289062,"Godfather, The (1972)",Crime|Drama,1972
2,2959,218,4.272936,Fight Club (1999),Action|Crime|Drama|Thriller,1999
3,50,204,4.237745,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,1995
4,260,251,4.231076,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,1977
5,527,220,4.225000,Schindler's List (1993),Drama|War,1993
6,1196,211,4.215640,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,1980
7,1198,200,4.207500,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,1981
8,296,307,4.197068,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,1994
9,2571,278,4.192446,"Matrix, The (1999)",Action|Sci-Fi|Thriller,1999


In [64]:
# Saving the data 
popular_movies_final.to_csv(r'..\datasets\new_data_sets\popular_movies.csv',index=False)

# Collaborative Filtering Recommender 

In [3]:
colab_filt_df = ratings.copy()

In [6]:
colab_filt_df['userId'].nunique()

610

In [14]:
colab_filt_df

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [67]:
# Getting only those users who have rated at least 30 movies 
x = colab_filt_df.groupby('userId')['rating'].count() > 30
# choosing users that specify the above condition and getting there id 
filtered_users = x[x].index

In [68]:
# now consider only filtered_users in ratings frame
filtered_ratings = ratings[ratings['userId'].isin(filtered_users)]

In [43]:
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [ ]:
print(filtered_ratings['userId'].nunique())
print(filtered_ratings['movieId'].nunique())

498
9689


In [82]:
# so we left with 498 users who rated over 9689
# Now we create our final collaborative filtering data frame

"""
Below function work like it place the movieId.unique() in first column and for every userId.unque() it create a column and the values of that columns are the rating the user provided to that movie

"""
final_colab_df = filtered_ratings.pivot_table(index = 'movieId',columns='userId',values='rating')
final_colab_df

userId,1,3,4,5,6,7,8,9,10,11,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,NaN,4.0,NaN,4.5,NaN,NaN,NaN,NaN,...,4.0,NaN,4.0,3.0,4.0,2.5,4.0,2.5,3.0,5.0
2,NaN,NaN,NaN,NaN,4.0,NaN,4.0,NaN,NaN,NaN,...,NaN,4.0,NaN,5.0,3.5,NaN,NaN,2.0,NaN,NaN
3,4.0,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN
4,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193585,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [83]:
# replacing missing value with 0
final_colab_df = final_colab_df.fillna(0,inplace=True)
final_colab_df

userId,1,3,4,5,6,7,8,9,10,11,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,0.0,4.0,0.0,4.5,0.0,0.0,0.0,0.0,...,4.0,0.0,4.0,3.0,4.0,2.5,4.0,2.5,3.0,5.0
2,0.0,0.0,0.0,0.0,4.0,0.0,4.0,0.0,0.0,0.0,...,0.0,4.0,0.0,5.0,3.5,0.0,0.0,2.0,0.0,0.0
3,4.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
4,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193583,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193585,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
